# Reproducing the tables

Every number in the paper's tables, printed from the files in this repository.

**Section 1** reads the summary CSVs under `results/`. It needs only pandas and runs in
seconds with no downloads.

**Section 2** recomputes the chemistry and the docking scores from the generated molecules
themselves, which is the honest check that the summaries were not typed by hand. It needs
`archive/chiediff_generated_molecules.pt` from the data archive:

```bash
python tools/download_archive.py --doi 10.5281/zenodo.23029439
```

Two things to keep in mind while reading the output.

**The cohorts are not interchangeable.** 9,999 molecules were sampled, 9,857 of them could be
reconstructed and exported to SDF, 8,340 docked successfully, and 8,069 are in the strict pose
cohort. The attrition is not
independent of the model, because fragmented molecules fail docking and the models that
fragment most lose most. Each table below prints its own `n`.

**Medians, not means.** Vina's `score_only` and `minimize` return penalties of order 1e6
kcal/mol for a pose that sits outside the box, so a mean over those columns is meaningless.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

pd.set_option('display.width', 200)
pd.set_option('display.max_columns', 60)
pd.set_option('display.float_format', lambda v: f'{v:.3f}')

ROOT = Path.cwd()
if not (ROOT / 'results').is_dir():          # also works from a subdirectory
    ROOT = ROOT.parent
RES = ROOT / 'results'
ARCHIVE = ROOT / 'archive'

OURS = 'ours_vina'          # this work, as the id column spells it
REFERENCE = 'reference'     # the crystal ligand


def load(rel):
    return pd.read_csv(RES / rel)


def show(df, columns, title, index='label'):
    # Print the named columns of df, in the order the paper's tables use.
    out = df[[index] + [c for c in columns if c in df.columns]].copy()
    print(f'\n{title}\n' + '-' * len(title))
    print(out.to_string(index=False))
    return out

## Section 1 — the tables, from the summary CSVs

### Vina scores, the score-to-dock gap, and diversity

`vina_score` is a single-point score of the generated pose, `vina_dock` is a full re-dock,
and the gap between them is how far the pose had to move to reach a Vina minimum. A model
that scores well and docks poorly has produced a pose that Vina likes where it stands but
that is not at a minimum.

In [2]:
main = load('comparison/f1_sbdd/main_table/main_table.csv')
_ = show(main.sort_values('rank'),
         ['n_pockets', 'n_mols',
          'vina_score_avg', 'vina_score_med', 'vina_dock_avg', 'vina_dock_med',
          'vinardo_score_avg', 'vinardo_score_med', 'vinardo_dock_avg', 'vinardo_dock_med'],
         'Vina and Vinardo, generated pose and re-docked, average and median')
_ = show(main.sort_values('rank'),
         ['cnnaff_score_avg', 'cnnaff_score_med', 'cnnaff_dock_avg', 'cnnaff_dock_med',
          'div_avg', 'div_med', 'ha_avg', 'ha_med'],
         'GNINA CNN affinity, diversity and high affinity')
print('\ncolumns available:', ', '.join(main.columns))


Vina and Vinardo, generated pose and re-docked, average and median
------------------------------------------------------------------
             label  n_pockets  n_mols  vina_score_avg  vina_score_med  vina_dock_avg  vina_dock_med  vinardo_score_avg  vinardo_score_med  vinardo_dock_avg  vinardo_dock_med
Reference (native)        100     100          -6.362          -6.463         -7.450         -7.263             -6.319             -6.308            -7.746            -7.356
        Pocket2Mol        100    9831          -5.139          -4.698         -7.152         -6.785             -4.193             -4.008            -6.893            -6.460
        TargetDiff        100    9036          -5.466          -6.302         -7.802         -7.908             -4.417             -5.189            -7.258            -7.380
          MolCRAFT        100    9667          -6.587          -7.044         -7.924         -8.006             -5.836             -6.230            -7.724            -7

### Redocking consistency

The score-to-dock gap and the redocking RMSD measured per engine on one cohort. Each engine
block comes from a single run, so the columns within a block are paired and the columns
across blocks are not.

In [3]:
redock = load('comparison/f2_pose_fidelity/redocking_consistency/redocking_consistency.csv')
for eng in ('vina', 'vinardo', 'gnina'):
    _ = show(redock, [f'{eng}_n_engine', f'{eng}_gap_avg', f'{eng}_gap_med',
                      f'{eng}_rmsd_avg', f'{eng}_rmsd_med', f'{eng}_rmsd_pct_lt2'],
             f'{eng}: gap (kcal/mol) and RMSD (A), with the share within 2 A')


vina: gap (kcal/mol) and RMSD (A), with the share within 2 A
------------------------------------------------------------
     label  vina_n_engine  vina_gap_avg  vina_gap_med  vina_rmsd_avg  vina_rmsd_med  vina_rmsd_pct_lt2
  Test set             98         1.307         0.697          5.149          5.630             24.490
Pocket2Mol           9741         2.382         1.934          4.742          4.342             26.773
TargetDiff           8921         2.198         1.190          4.581          4.648             28.797
    IPDiff           8908         2.011         1.041          4.586          4.724             29.165
   AliDiff           9121         1.846         0.667          4.145          3.925             36.915
  MolCRAFT           9489         1.577         0.665          4.251          3.919             36.685
    PIDiff           8536         1.760         0.868          4.366          4.269             31.596
    KGDiff           8606         1.379         0.512

### Calculated molecular properties

QED, synthetic accessibility, LogP, TPSA, molecular weight and the Lipinski rule count.
`n_unmeasurable` is the number of molecules RDKit could not sanitize, kept in the
denominator rather than dropped.

In [4]:
prop = load('comparison/f1_sbdd/properties/property_overall.csv')
_ = show(prop, ['n_mols', 'n_unmeasurable', 'qed_med', 'sa_med', 'logp_med',
                'tpsa_med', 'mw_med', 'lipinski_med'],
         'Calculated properties, median per molecule')


Calculated properties, median per molecule
------------------------------------------
             label  n_mols  n_unmeasurable  qed_med  sa_med  logp_med  tpsa_med  mw_med  lipinski_med
Reference (native)     100               0    0.468   0.740     1.090    92.500 320.600         5.000
        Pocket2Mol    9831               0    0.577   0.760     1.330    60.700 208.300         5.000
        TargetDiff    9036               0    0.481   0.580     1.350    93.600 348.100         5.000
            IPDiff    9012               5    0.524   0.590     3.280    54.400 336.400         5.000
          MolCRAFT    9667               0    0.506   0.680     1.280    92.500 325.400         5.000
           AliDiff    9236               4    0.512   0.560     4.470    40.500 344.500         5.000
PIDiff (retrained)    8654               0    0.438   0.550     1.280    91.300 340.400         5.000
            KGDiff    8813               0    0.508   0.540     2.780    63.200 343.500         5

### Ring sizes

Occupancy is the share of all rings that have a given size. A generative model that favours
4- and 7-membered rings is producing chemistry that does not look like the training
distribution, which the bond-distance JSD below measures differently.

In [5]:
ring = load('comparison/f1_sbdd/ring_size/ring_size_overall.csv')
_ = show(ring, ['n_mols', 'mean_rings_per_mol', 'acyclic_pct'] +
         [f'ring{n}_occ_pct' for n in range(3, 10)] + ['ring_ge10_share_pct'],
         'Ring-size occupancy (% of all rings)')


Ring-size occupancy (% of all rings)
------------------------------------
             label  n_mols  mean_rings_per_mol  acyclic_pct  ring3_occ_pct  ring4_occ_pct  ring5_occ_pct  ring6_occ_pct  ring7_occ_pct  ring8_occ_pct  ring9_occ_pct  ring_ge10_share_pct
Reference (native)     100               2.470       13.000          1.700          0.000         30.200         67.400          0.800          0.000          0.000                2.000
        Pocket2Mol    9831               2.845        5.500          0.100          0.000         16.400         80.400          2.600          0.300          0.100                0.700
        TargetDiff    9036               2.868        7.100          0.000          2.800         30.800         50.700         12.100          2.700          0.900                3.500
            IPDiff    9012               3.366        2.800          0.000          2.200         22.700         52.000         15.900          5.300          2.000                4

### Bond-geometry Jensen-Shannon distances

`ejsd` is the distance between a model's bond-length distributions and the reference's,
averaged over bond types. Lower is closer to real chemistry.

In [6]:
jsd = load('comparison/f1_sbdd/bond_jsd/bond_jsd_overall.csv')
_ = show(jsd, ['n_measured', 'n_unsanitizable', 'ejsd', 'jsd_C-C', 'jsd_C=C',
               'jsd_C:C', 'jsd_C-N', 'jsd_C-O', 'jsd_atom'],
         'Bond-length and atom-type JSD against the reference')


Bond-length and atom-type JSD against the reference
---------------------------------------------------
             label  n_measured  n_unsanitizable  ejsd  jsd_C-C  jsd_C=C  jsd_C:C  jsd_C-N  jsd_C-O  jsd_atom
Reference (native)         100                0   NaN      NaN      NaN      NaN      NaN      NaN     0.069
        Pocket2Mol        9831                0 0.461    0.496    0.561    0.416    0.425    0.454     0.092
        TargetDiff        9036                0 0.378    0.369    0.505    0.263    0.363    0.421     0.060
            IPDiff        9007                5 0.430    0.452    0.530    0.221    0.411    0.489     0.150
          MolCRAFT        9667                0 0.333    0.321    0.495    0.323    0.301    0.354     0.061
           AliDiff        9232                4 0.414    0.393    0.569    0.333    0.416    0.500     0.203
PIDiff (retrained)        8654                0 0.506    0.530    0.556    0.369    0.442    0.533     0.064
            KGDiff     

### Redocking RMSD

How far the generated pose moves when an engine re-docks it. `*_pct_lt2` is the share of
molecules that stay within 2 A. `clash` and `strain` are PoseCheck's, on the same cohort.

In [7]:
heavy = load('pose_fidelity/heavy_clash/table_inputs.csv')
print('PoseCheck clash, two definitions. The paper prints the heavy-atom-only count.')
print('-' * 72)
print(heavy[['id', 'table', 'cohort', 'n', 'clash_h_incl', 'heavy_clash_avg',
             'heavy_clash_ion_avg']].to_string(index=False))
print('\nclash_h_incl counts hydrogens on both sides and is 5 to 12 times larger, so the')
print('two are not interchangeable. heavy_clash_avg is the reported column.')

pose = load('comparison/f2_pose_fidelity/pose_stability/pose_stability_strict.csv')
_ = show(pose.sort_values('rank'),
         ['n_strict', 'vina_pct_lt2', 'vina_med', 'vinardo_pct_lt2', 'vinardo_med',
          'gnina_pct_lt2', 'gnina_med', 'clash_med', 'strain_med'],
         'Redocking RMSD (A) and PoseCheck, strict pose cohort')

eng = load('comparison/f2_pose_fidelity/pose_stability/pose_stability_engineonly.csv')
_ = show(eng.sort_values('rank'), ['n_strict', 'vina_pct_lt2', 'vina_med'],
         'Sensitivity: the same, on the RMSD-complete cohort instead')

PoseCheck clash, two definitions. The paper prints the heavy-atom-only count.
------------------------------------------------------------------------
                 id             table                   cohort    n  clash_h_incl  heavy_clash_avg  heavy_clash_ion_avg
          reference       tab:quality            table2_strict  100         7.980            0.520                0.380
         pocket2mol       tab:quality            table2_strict 9801         8.338            1.148                0.186
         targetdiff       tab:quality            table2_strict 8954        13.821            1.714                0.394
           molcraft       tab:quality            table2_strict 9657         9.672            1.008                0.422
     pidiff_retrain       tab:quality            table2_strict 8469         6.536            0.775                0.340
             ipdiff       tab:quality            table2_strict 8689        12.700            1.571                0.370
         

### PoseBusters physical validity

Each column is the share of molecules passing one check. `pass_all_pct` requires every
check at once, so it is dominated by whichever check is hardest.

In [8]:
pb = load('comparison/appendix/posebusters/pose_quality_summary_posebusters.csv')
_ = show(pb, ['subset', 'n_mol', 'pass_all_pct', 'all_atoms_connected', 'bond_lengths',
              'bond_angles', 'internal_steric_clash', 'internal_energy',
              'minimum_distance_to_protein'],
         'PoseBusters pass rates', index='model')


PoseBusters pass rates
----------------------
              model             subset  n_mol  pass_all_pct  all_atoms_connected  bond_lengths  bond_angles  internal_steric_clash  internal_energy  minimum_distance_to_protein
  Ours (Vina+Guide) docking_successful   8271        35.150              100.000        63.570       56.670                 80.200           92.520                       98.150
  Ours (Vina+Guide)       all_exported   9782        29.720               84.550        60.900       55.790                 75.370           90.610                       93.100
  Ours (Guide Only) docking_successful   2042        30.610              100.000        89.860       39.370                 81.930           87.810                       97.360
  Ours (Guide Only)       all_exported   2042        30.610              100.000        89.860       39.370                 81.930           87.810                       97.360
         TargetDiff docking_successful   9025        51.650         

### PoseBusters clash counts

The boolean above says whether a pose clashes at all. This keeps the count of clashing
heavy-atom pairs, which separates a pose that grazes the protein from one that is inside it.

In [9]:
pbd = load('comparison/f2_pose_fidelity/pb_distance/pb_distance_strict.csv')
_ = show(pbd, ['cohort', 'n', 'clash_free_protein_pct', 'pb_clashes_protein_med',
               'clash_free_all_pct', 'within_5A_pct'],
         'PoseBusters intermolecular distance, strict pose cohort')


PoseBusters intermolecular distance, strict pose cohort
-------------------------------------------------------
             label        cohort    n  clash_free_protein_pct  pb_clashes_protein_med  clash_free_all_pct  within_5A_pct
           AliDiff table2_strict 8858                  93.310                   0.000              92.460        100.000
            IPDiff table2_strict 8689                  87.790                   0.000              86.870        100.000
            KGDiff table2_strict 8278                  97.550                   0.000              96.670        100.000
          MolCRAFT table2_strict 9657                  96.190                   0.000              94.590        100.000
Reference (native) table2_strict  100                  98.000                   0.000              98.000        100.000
 Ours (Guide Only) table2_strict 1920                  97.240                   0.000              95.990        100.000
            PIDiff table2_strict  831   

### Non-covalent interactions, measured twice

PLIP and ProLIF are independent implementations and they disagree on absolute counts while
agreeing on the ranking. ProLIF here is pinned to 2.2.0: version 2.1.0 changed the
hydrophobic SMARTS definition, and the same crystal ligands give 2.04 hydrophobic contacts
per molecule under the older definition against 1.44 under the newer one. So these ProLIF
numbers are not comparable with ProLIF numbers from other papers, and the PLIP columns are
the ones to use for that.

In [10]:
nci = load('comparison/f3_nci/nci_summary_strict.csv')
_ = show(nci, ['n', 'heavy_mean', 'plip_n_hbond', 'plip_n_hydrophobic', 'plip_n_pistack',
               'plip_total', 'plip_per_heavy'],
         'PLIP interaction counts per molecule')
_ = show(nci, ['n', 'prolif_HBAcceptor', 'prolif_HBDonor', 'prolif_Hydrophobic',
               'prolif_VdWContact', 'prolif_total', 'prolif_per_heavy'],
         'ProLIF 2.2.0 interaction counts per molecule')


PLIP interaction counts per molecule
------------------------------------
             label    n  heavy_mean  plip_n_hbond  plip_n_hydrophobic  plip_n_pistack  plip_total  plip_per_heavy
Reference (native)  100      22.750         3.920               3.600           0.290       8.450           0.392
 Ours (Vina+Guide) 8101      24.233         4.067               6.977           0.273      11.749           0.504
 Ours (Guide Only) 1925      23.851         3.784               5.567           0.214       9.920           0.434
        TargetDiff 8954      24.138         4.090               4.797           0.246       9.643           0.417
            KGDiff 8280      24.012         3.494               5.994           0.210      10.086           0.437
PIDiff (retrained) 8476      23.913         3.321               4.249           0.192       8.226           0.360
            IPDiff 8705      23.825         2.744               6.208           0.258       9.512           0.413
  Ours (Vina 

### Pocket specificity

For each molecule, the Vina dock score in its own pocket against the same molecule docked
into an unrelated pocket. A model that has learned the pocket should lose affinity off
target.

The reported figures come from the pre-aggregated table, because the aggregation is not
incidental: the off-target column carries out-of-box penalties of order 1e6 kcal/mol, so it
is clipped to [-20, 20] and then averaged within each pocket before the pockets are
averaged. The per-molecule file is re-aggregated below as a check, and the two agree only
when the same rule is applied.

In [11]:
master = load('comparison/master_table.csv')
_ = show(master, ['f4_n_pk', 'f4_n_mol', 'f4_on_med', 'f4_off_med',
                  'f4_delta_p_avg', 'f4_delta_p_med',
                  'f4_win_pct_avg', 'f4_win_pct_med'],
         'Pocket specificity as reported: Vina dock on and off target (kcal/mol)')


Pocket specificity as reported: Vina dock on and off target (kcal/mol)
----------------------------------------------------------------------
                      label  f4_n_pk  f4_n_mol  f4_on_med  f4_off_med  f4_delta_p_avg  f4_delta_p_med  f4_win_pct_avg  f4_win_pct_med
         Reference (native)   99.000    99.000     -7.202      -6.960           0.778           0.240          58.590         100.000
                 Pocket2Mol   99.000  9731.000     -6.942      -6.921           0.241           0.186          50.400          64.000
                 TargetDiff   99.000  8992.000     -7.989      -7.416           0.624           0.434          56.730          67.530
                     IPDiff   99.000  8994.000     -8.688      -8.071           0.828           0.311          56.330          66.670
                   MolCRAFT   99.000  9573.000     -7.922      -7.377           0.628           0.403          58.330          68.000
                    AliDiff   99.000  9188.000     -8

In [12]:
delta = pd.read_csv(RES / 'comparison/f4_delta_score/delta_score_per_mol.csv.gz')
print('per-molecule columns:', ', '.join(delta.columns))

d = delta.dropna(subset=['on_dock', 'off_dock']).copy()

# Every model is restricted to the pockets this work has, so the column compares the same
# targets for everyone. Without it a model measured on one extra pocket is not comparable,
# and the published numbers do not reproduce.
COMMON = set(d.loc[d.model == 'vina_fixed_on', 'pocket_idx'].unique())
d = d[d.pocket_idx.isin(COMMON)].copy()
print(f'common pocket set: {len(COMMON)} pockets')

# Winsorise both sides. An off-target pose placed outside its box earns a Vina penalty of
# order 1e6 kcal/mol, which no real ligand approaches, and leaving those in moves a model
# by about a kcal/mol. The bound barely matters between 100 and 20.
d['on_c'] = d['on_dock'].clip(-20, 20)
d['off_c'] = d['off_dock'].clip(-20, 20)
d['delta'] = d['off_c'] - d['on_c']

# Reduce each pocket to its mean first, then combine across pockets: the pocket is the
# independent unit, because every molecule in a pocket faces the same off-target protein.
per_pocket = d.groupby(['model', 'pocket_idx']).agg(
    delta=('delta', 'mean'), on=('on_c', 'mean'), off=('off_c', 'mean')).reset_index()
recomputed = per_pocket.groupby('model').agg(
    n_pockets=('delta', 'size'), delta_avg=('delta', 'mean'),
    delta_med=('delta', 'median'), on_med=('on', 'median'), off_med=('off', 'median'))
print('\nRecomputed from the per-molecule file, on the common pockets, clipped to [-20, 20]')
print('-' * 80)
print(recomputed.sort_values('delta_avg', ascending=False).to_string())

ours = master[master.id == OURS]
if not ours.empty:
    row = ours.iloc[0]
    mine = recomputed.loc['vina_fixed_on'] if 'vina_fixed_on' in recomputed.index else None
    print('\nthis work, reported against recomputed:')
    for label, col, key in [('delta avg', 'f4_delta_p_avg', 'delta_avg'),
                            ('delta med', 'f4_delta_p_med', 'delta_med'),
                            ('on  med  ', 'f4_on_med', 'on_med'),
                            ('off med  ', 'f4_off_med', 'off_med')]:
        line = f'  {label}  reported {float(row[col]):8.4f}'
        if mine is not None:
            line += f'   recomputed {mine[key]:8.4f}'
        print(line)

# Every row of the published table, checked rather than eyeballed.
PUB = {'vina_fixed_on': 'ours_vina', 'vina_fixed_noguide': 'ours_noguide',
       'kgdiff': 'kgdiff', 'reference': 'reference', 'alidiff': 'alidiff',
       'targetdiff': 'targetdiff', 'ipdiff': 'ipdiff', 'molcraft': 'molcraft',
       'pocket2mol': 'pocket2mol', 'ar': 'ar', 'decompdiff': 'decompdiff',
       'pidiff_retrain': 'pidiff_retrain'}
checked = disagree = 0
for model, pub_id in PUB.items():
    if model not in recomputed.index:
        continue
    pr = master[master.id == pub_id]
    if pr.empty or not str(pr.iloc[0]['f4_delta_p_avg']).strip():
        continue
    pr, mr = pr.iloc[0], recomputed.loc[model]
    for col, key in [('f4_n_pk', 'n_pockets'), ('f4_delta_p_avg', 'delta_avg'),
                     ('f4_delta_p_med', 'delta_med'), ('f4_on_med', 'on_med'),
                     ('f4_off_med', 'off_med')]:
        checked += 1
        if abs(float(pr[col]) - float(mr[key])) > 5e-4:
            disagree += 1
            print(f'  DISAGREES {pub_id} {col}: {pr[col]} against {mr[key]:.4f}')
print(f'\n{checked} published cells checked, {disagree} disagree')

per-molecule columns: model, pocket_idx, mol_index, off_pocket_idx, smiles, on_score, on_min, on_dock, off_score, off_min, off_dock, delta_dock_sbe, box_expanded
common pocket set: 99 pockets

Recomputed from the per-molecule file, on the common pockets, clipped to [-20, 20]
--------------------------------------------------------------------------------
                    n_pockets  delta_avg  delta_med  on_med  off_med
model                                                               
vina_fixed_on              99      2.034      1.756  -9.936   -7.966
vina_fixed_noguide         99      1.632      1.419  -8.951   -7.581
kgdiff                     99      1.521      1.034  -9.264   -8.134
alidiff                    99      1.259      0.863  -8.816   -7.809
pidiff_retrain             99      0.847      0.557  -8.249   -7.720
ipdiff                     99      0.828      0.311  -8.688   -8.071
reference                  99      0.778      0.240  -7.202   -6.960
molcraft              

### Ablations

The guidance and energy-term arms, measured with both interaction instruments.

In [13]:
nci_abl = load('diagnostics/ablation_nci/nci_summary_abl.csv')
_ = show(nci_abl.sort_values(['guidance', 'label']),
         ['guidance', 'tag', 'n_plip', 'plip_hbond', 'plip_hydrophobic', 'plip_total',
          'plip_per_heavy'],
         'PLIP interaction counts per molecule, by energy term and guidance')
_ = show(nci_abl.sort_values(['guidance', 'label']),
         ['guidance', 'n_prolif', 'prolif_hbacceptor', 'prolif_hbdonor',
          'prolif_hydrophobic_contact', 'prolif_total', 'posecheck_clash_mean',
          'posecheck_strain_median'],
         'ProLIF 2.2.0 and PoseCheck, same arms')

yld = load('diagnostics/ablation_nci_v2/dock_success.csv')
_ = show(yld, ['n_budget', 'n_generated', 'n_docked', 'dock_success_pct'],
         'Docking yield per arm', index='id')

inter = load('diagnostics/ablation_nci_v2/plip_intersection.csv')
_ = show(inter, ['n', 'n_plip_paired', 'hbond', 'hydrophobic', 'total', 'per_heavy'],
         'PLIP re-pooled on the molecules both instruments measured', index='report_id')
print('\nThis is a different cohort from the PLIP block above, which pools over every')
print('PLIP-measured molecule, so the two disagree. The paper prints this one.')

abl = load('diagnostics/ablation_nci_v2/posecheck_ablation_all.csv')
print('\n\nProLIF / PoseCheck ablation, full per-arm detail')
print('-' * 47)
print(abl.to_string(index=False))


PLIP interaction counts per molecule, by energy term and guidance
-----------------------------------------------------------------
                      label  guidance                     tag  n_plip  plip_hbond  plip_hydrophobic  plip_total  plip_per_heavy
arm1 no-physics (novdw) OFF       OFF            abl_none_off    2098       3.740             4.360       8.860           0.391
           arm2 vdW (S) OFF       OFF             abl_vdw_off    2159       3.680             5.480       9.890           0.426
        arm3 h-bond (H) OFF       OFF           abl_hbond_off    1402       3.440             4.710       8.740           0.429
   arm4 hydrophobic (P) OFF       OFF           abl_hydro_off    2225       4.000             5.060       9.830           0.426
  arm5 vdW+h-bond (S+H) OFF       OFF       abl_vdw_hbond_off    2085       4.220             5.550      10.490           0.453
   arm6 vdW+hydro (S+P) OFF       OFF       abl_vdw_hydro_off    2071       3.590             5.790

## Section 2 — recomputed from the molecules

This loads the generated molecules and recomputes the chemistry and the docking statistics
from scratch, so the summary rows above can be checked against the objects they summarise.

The `.pt` holds one list per pocket. Each entry carries the RDKit molecule, its SMILES, the
reference ligand filename, the `chem_results` dict the sampler computed, and the Vina
`score_only` / `minimize` / `dock` results. An entry whose molecule could not be
reconstructed is kept with `mol` set to `None`, which is what keeps a pocket's denominator
at 100.

In [14]:
import sys

sys.path.insert(0, str(ROOT))

PT = ARCHIVE / 'chiediff_generated_molecules.pt'
HAVE_PT = PT.is_file()
if not HAVE_PT:
    print(f'{PT} not found.\n'
          'Section 2 needs the data archive:\n'
          '    python tools/download_archive.py --doi 10.5281/zenodo.23029439')
else:
    import torch
    from rdkit import RDLogger

    RDLogger.DisableLog('rdApp.*')
    try:                                  # torch < 1.13 has no weights_only
        grouped = torch.load(PT, map_location='cpu', weights_only=False)
    except TypeError:
        grouped = torch.load(PT, map_location='cpu')
    flat = [m for pocket in grouped for m in pocket]
    print(f'{len(grouped)} pockets, {len(flat)} entries, '
          f'{sum(1 for m in flat if m.get("mol") is not None)} with a molecule')

/home/ktori1361/ChIEDiff/archive/chiediff_generated_molecules.pt not found.
Section 2 needs the data archive:
    python tools/download_archive.py --doi 10.5281/zenodo.23029439


### The cohort chain, counted

Each line is a denominator used somewhere in the paper. They are produced here rather than
quoted, because the step from one to the next is where molecules are lost.

In [15]:
if HAVE_PT:
    def docked(m):
        v = m.get('vina') or {}
        d = v.get('dock')
        return bool(d) and d[0].get('affinity') is not None

    n_attempted = len(flat)
    n_mol = sum(1 for m in flat if m.get('mol') is not None)
    n_chem = sum(1 for m in flat if m.get('chem_results'))
    n_dock = sum(1 for m in flat if docked(m))
    for label, n in [('entries in the file', n_attempted),
                     ('reconstructed to a molecule', n_mol),
                     ('with chem_results', n_chem),
                     ('docking-successful', n_dock)]:
        print(f'{label:32s} {n:6d}')

### Chemistry, recomputed

QED, SA and LogP come straight out of `chem_results`, which is what the sampler stored and
what the property table above aggregates. Recomputing them from the exported SDFs instead
gives slightly different values for about 3.5% of molecules, because writing a molecule with
unusual valences to SDF and reading it back lets RDKit re-perceive the valence and turn
implicit hydrogens into radical electrons. `scripts/chem_props.py` documents that.

In [16]:
if HAVE_PT:
    rows = []
    for pi, pocket in enumerate(grouped):
        for m in pocket:
            c = m.get('chem_results')
            if not c:
                continue
            sizes = {int(k): v for k, v in (c.get('ring_size') or {}).items()}
            rows.append({'pocket': pi, 'qed': c['qed'], 'sa': c['sa'], 'logp': c['logp'],
                         'lipinski': int(c['lipinski']),
                         'heavy': m['mol'].GetNumHeavyAtoms() if m.get('mol') else np.nan,
                         **{f'ring{n}': int(sizes.get(n, 0)) for n in range(3, 10)}})
    chem = pd.DataFrame(rows)
    print(f'n = {len(chem)}')
    print(chem[['qed', 'sa', 'logp', 'lipinski', 'heavy']].median().to_frame('median').T
          .to_string(index=False))

    ring_counts = chem[[f'ring{n}' for n in range(3, 10)]].sum()
    occ = 100 * ring_counts / ring_counts.sum()
    print('\nring-size occupancy (%)')
    print(occ.round(1).to_string())

    published = prop[prop.id == OURS]
    if not published.empty:
        p = published.iloc[0]
        print('\nagainst the published row:')
        for k in ('qed', 'sa', 'logp'):
            print(f'  {k:5s} recomputed {chem[k].median():7.3f}   published {p[k + "_med"]:7.3f}')

### Vina, recomputed

Medians over the docking-successful molecules, which is the population every Vina column in
the paper aggregates.

In [17]:
if HAVE_PT:
    rec = []
    for pi, pocket in enumerate(grouped):
        for m in pocket:
            v = m.get('vina') or {}
            if not docked(m):
                continue
            def first(key):
                arr = v.get(key)
                return arr[0]['affinity'] if arr else np.nan
            rec.append({'pocket': pi, 'score': first('score_only'),
                        'minimize': first('minimize'), 'dock': first('dock')})
    vina = pd.DataFrame(rec)
    vina['gap'] = vina['score'] - vina['dock']
    print(f'n = {len(vina)} docking-successful molecules')
    print(vina[['score', 'minimize', 'dock', 'gap']].median().to_frame('median').T
          .to_string(index=False))
    print('\nmeans for comparison, which the out-of-box penalties make unusable:')
    print(vina[['score', 'minimize', 'dock']].mean().to_frame('mean').T.to_string(index=False))

    published = main[main.id == OURS]
    if not published.empty:
        p = published.iloc[0]
        print('\nagainst the published row:')
        for k, col in [('score', 'vina_score_med'), ('dock', 'vina_dock_med')]:
            if col in p:
                print(f'  {k:5s} recomputed {vina[k].median():7.3f}   '
                      f'published {p[col]:7.3f}')

### Diversity and high affinity

Diversity is the mean pairwise Tanimoto distance over Morgan fingerprints within a pocket,
so it is computed per pocket and then averaged. High affinity is the share of generated
molecules that beat the crystal ligand in the same pocket, which needs the reference
docking scores and is therefore reported only when they are present.

In [18]:
if HAVE_PT:
    from rdkit import Chem
    from rdkit.Chem import DataStructs

    # The published diversity column is the mean pairwise Tanimoto DISTANCE over RDKit
    # topological fingerprints (Chem.RDKFingerprint, fpSize 2048), among the
    # docking-successful molecules of one pocket, averaged over pockets. A Morgan
    # fingerprint gives a visibly different number, so the choice matters.
    per_pocket = []
    for pocket in grouped:
        fps = []
        for m in pocket:
            if m.get('mol') is None or not docked(m):
                continue
            try:
                fps.append(Chem.RDKFingerprint(m['mol']))
            except Exception:
                pass
        if len(fps) < 2:
            continue
        total = n = 0
        for i in range(len(fps) - 1):
            sims = DataStructs.BulkTanimotoSimilarity(fps[i], fps[i + 1:])
            total += sum(1.0 - s for s in sims)
            n += len(sims)
        per_pocket.append(total / n)

    print(f'diversity over {len(per_pocket)} pockets: '
          f'mean {np.mean(per_pocket):.3f}, median {np.median(per_pocket):.3f}')

    published = main[main.id == OURS]
    if not published.empty:
        p = published.iloc[0]
        print(f'published:                      '
              f'mean {p["div_avg"]:.3f}, median {p["div_med"]:.3f}')
        print(f'\nhigh affinity (share beating the crystal ligand), published: '
              f'mean {p["ha_avg"]:.1f}%, median {p["ha_med"]:.1f}%')
        print('Recomputing high affinity needs the reference docking scores, which are in '
              'the\nreference model\'s own .pt rather than this one.')

---

If a recomputed number disagrees with its published row by more than rounding, that is worth
reporting as an issue on the repository rather than reconciling by hand. The summaries and
the molecules come from the same measurement run, so they should agree.